# Evaluation Matrices (Section VI.C / VII)
Each cell corresponds to a matrix in the evaluation checklist.


In [1]:
from __future__ import annotations
import json, math, re, statistics
from pathlib import Path
from typing import Dict, Any, List, Tuple

RESULTS_ROOT = Path('results')
DATASETS = {
    'wiki2018': RESULTS_ROOT / 'wiki2018',
    'wikipedia_september_2007': RESULTS_ROOT / 'wikipedia_september_2007',
}

METHODS_MAIN = [
    'baseline_lru',
    'baseline_lru-k',
    'ilnse_xu',
    'gdbt_A2',
    'ilnse_A2_fixed',
    'ilnse_A2_drift_only',
    'ilnse_A2_guard_only',
    'ilnse_A2_full',
    'ilnse_A0_full',
]

ABLAT = ['ilnse_A2_fixed', 'ilnse_A2_drift_only', 'ilnse_A2_guard_only', 'ilnse_A2_full']

def _rankdata(x: List[float]) -> List[float]:
    order = sorted(range(len(x)), key=lambda i: x[i])
    ranks = [0.0]*len(x)
    i = 0
    while i < len(order):
        j = i + 1
        while j < len(order) and x[order[j]] == x[order[i]]:
            j += 1
        avg = (i + 1 + j) / 2.0
        for k in range(i, j):
            ranks[order[k]] = avg
        i = j
    return ranks

def spearman(x: List[float], y: List[float]) -> float | None:
    if len(x) != len(y) or len(x) < 2:
        return None
    rx = _rankdata(x); ry = _rankdata(y)
    mx = sum(rx)/len(rx); my = sum(ry)/len(ry)
    num = sum((a-mx)*(b-my) for a,b in zip(rx,ry))
    denx = math.sqrt(sum((a-mx)**2 for a in rx))
    deny = math.sqrt(sum((b-my)**2 for b in ry))
    if denx == 0 or deny == 0:
        return None
    return num / (denx*deny)

def load_latest_summaries(dataset_dir: Path) -> Dict[Tuple[str,int], Dict[str,Any]]:
    pat = re.compile(r'^(\d{3})_summary_(.+)_(\d+)\.json$')
    records: Dict[Tuple[str,int], Tuple[int, Dict[str,Any]]] = {}
    for p in dataset_dir.glob('*_summary_*.json'):
        if p.name.endswith('_all_sizes.json'):
            continue
        m = pat.match(p.name)
        if not m:
            continue
        run_id = int(m.group(1))
        model = m.group(2)
        cache = int(m.group(3))
        key = (model, cache)
        cur = records.get(key)
        if cur is None or run_id > cur[0]:
            records[key] = (run_id, json.loads(p.read_text()))
    return {k: v[1] for k,v in records.items()}

def iter_jsonl(path: Path):
    with path.open('r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            yield json.loads(line)

def get_caches(records):
    return sorted({c for (_,c) in records})


In [2]:
# Diagnostics: verify data availability
for ds_name, ds_dir in DATASETS.items():
    print(f'Dataset {ds_name}:', ds_dir)
    if not ds_dir.exists():
        print('  -> MISSING directory')
        continue
    files = list(ds_dir.glob('*_summary_*.json'))
    print('  summary files:', len(files))
    records = load_latest_summaries(ds_dir)
    print('  latest summaries loaded:', len(records))
    models = sorted({m for (m, _) in records})
    print('  models:', models)
    caches = get_caches(records)
    print('  cache sizes:', caches)
    print()


Dataset wiki2018: results/wiki2018
  summary files: 59
  latest summaries loaded: 54
  models: ['baseline_lru', 'baseline_lru-k', 'gdbt_A2', 'ilnse_A0_full', 'ilnse_A2_drift_only', 'ilnse_A2_fixed', 'ilnse_A2_full', 'ilnse_A2_guard_only', 'ilnse_xu']
  cache sizes: [16123, 20154, 40308, 60463, 80617, 100772]

Dataset wikipedia_september_2007: results/wikipedia_september_2007
  summary files: 59
  latest summaries loaded: 54
  models: ['baseline_lru', 'baseline_lru-k', 'gdbt_A2', 'ilnse_A0_full', 'ilnse_A2_drift_only', 'ilnse_A2_fixed', 'ilnse_A2_full', 'ilnse_A2_guard_only', 'ilnse_xu']
  cache sizes: [12221, 15277, 30554, 45831, 61109, 76386]



## (1) HR vs Cache Size (headline)


In [4]:
for ds_name, ds_dir in DATASETS.items():
    records = load_latest_summaries(ds_dir)
    caches = get_caches(records)
    print('DATASET:', ds_name)
    header = ['Method'] + [str(c) for c in caches]
    print('| ' + ' | '.join(header) + ' |')
    print('| ' + ' | '.join(['---']*len(header)) + ' |')
    for m in METHODS_MAIN:
        row = [m]
        for c in caches:
            val = records.get((m,c), {}).get('hit_ratio')
            row.append(f'{val:.6f}' if val is not None else 'NA')
        print('| ' + ' | '.join(row) + ' |')


DATASET: wiki2018
| Method | 16123 | 20154 | 40308 | 60463 | 80617 | 100772 |
| --- | --- | --- | --- | --- | --- | --- |
| baseline_lru | 0.392076 | 0.404896 | 0.440820 | 0.469151 | 0.499158 | 0.531905 |
| baseline_lru-k | 0.411472 | 0.423219 | 0.468433 | 0.511392 | 0.541176 | 0.560817 |
| ilnse_xu | 0.414850 | 0.438246 | 0.514731 | 0.550779 | 0.574605 | 0.591063 |
| gdbt_A2 | 0.457613 | 0.473261 | 0.523953 | 0.550556 | 0.564494 | 0.570360 |
| ilnse_A2_fixed | 0.463234 | 0.475633 | 0.518416 | 0.548435 | 0.573825 | 0.590889 |
| ilnse_A2_drift_only | 0.464642 | 0.477335 | 0.517919 | 0.543846 | 0.563862 | 0.576003 |
| ilnse_A2_guard_only | 0.467811 | 0.482733 | 0.526083 | 0.558826 | 0.584642 | 0.605156 |
| ilnse_A2_full | 0.466867 | 0.480824 | 0.526211 | 0.559610 | 0.585296 | 0.605636 |
| ilnse_A0_full | 0.434173 | 0.467405 | 0.525184 | 0.562923 | 0.587955 | 0.606730 |
DATASET: wikipedia_september_2007
| Method | 12221 | 15277 | 30554 | 45831 | 61109 | 76386 |
| --- | --- | --- | --- | -

## (2) Normalized Gain vs LRU and vs Fixed‑Budget


In [7]:
BASELINES = ['baseline_lru', 'ilnse_A2_fixed']
for ds_name, ds_dir in DATASETS.items():
    records = load_latest_summaries(ds_dir)
    caches = get_caches(records)
    print('DATASET:', ds_name)
    for b in BASELINES:
        if (b, caches[0]) not in records:
            continue
        header = ['Method'] + [str(c) for c in caches]
        print(f'ΔHR vs {b}')
        print('| ' + ' | '.join(header) + ' |')
        print('| ' + ' | '.join(['---']*len(header)) + ' |')
        for m in METHODS_MAIN:
            row = [m]
            for c in caches:
                v = records.get((m,c), {}).get('hit_ratio')
                bv = records.get((b,c), {}).get('hit_ratio')
                d = (v - bv) if (v is not None and bv is not None) else None
                row.append(f'{d:.6f}' if d is not None else 'NA')
            print('| ' + ' | '.join(row) + ' |')


DATASET: wiki2018
ΔHR vs baseline_lru
| Method | 16123 | 20154 | 40308 | 60463 | 80617 | 100772 |
| --- | --- | --- | --- | --- | --- | --- |
| baseline_lru | 0.000000 | 0.000000 | 0.000000 | 0.000000 | 0.000000 | 0.000000 |
| baseline_lru-k | 0.019397 | 0.018322 | 0.027613 | 0.042240 | 0.042018 | 0.028911 |
| ilnse_xu | 0.022774 | 0.033349 | 0.073910 | 0.081628 | 0.075447 | 0.059158 |
| gdbt_A2 | 0.065537 | 0.068364 | 0.083133 | 0.081405 | 0.065336 | 0.038455 |
| ilnse_A2_fixed | 0.071158 | 0.070737 | 0.077596 | 0.079284 | 0.074667 | 0.058983 |
| ilnse_A2_drift_only | 0.072567 | 0.072439 | 0.077099 | 0.074695 | 0.064704 | 0.044098 |
| ilnse_A2_guard_only | 0.075735 | 0.077837 | 0.085263 | 0.089675 | 0.085484 | 0.073250 |
| ilnse_A2_full | 0.074791 | 0.075928 | 0.085390 | 0.090458 | 0.086138 | 0.073731 |
| ilnse_A0_full | 0.042097 | 0.062509 | 0.084364 | 0.093771 | 0.088797 | 0.074825 |
ΔHR vs ilnse_A2_fixed
| Method | 16123 | 20154 | 40308 | 60463 | 80617 | 100772 |
| --- | --- | --- 

## (3) Admission Intensity Summary (avg M_t, |A_t|, admit rate)


In [8]:
for ds_name, ds_dir in DATASETS.items():
    records = load_latest_summaries(ds_dir)
    caches = get_caches(records)
    print('DATASET:', ds_name)
    header = ['Method','Cache','avg_M','avg_A','admit_rate','miss_rate_avg']
    print('| ' + ' | '.join(header) + ' |')
    print('| ' + ' | '.join(['---']*len(header)) + ' |')
    for m in METHODS_MAIN:
        for c in caches:
            s = records.get((m,c), {})
            cache_slots = s.get('cache_slots') or 0
            admit_budget_total = s.get('admit_budget_total')
            admit_selected_total = s.get('admit_selected_total')
            miss_candidates_total = s.get('miss_candidates_total')
            avg_M = (admit_budget_total/cache_slots) if (admit_budget_total is not None and cache_slots) else None
            avg_A = (admit_selected_total/cache_slots) if (admit_selected_total is not None and cache_slots) else None
            admit_rate = (admit_selected_total/miss_candidates_total) if (admit_selected_total is not None and miss_candidates_total) else None
            row = [m, c, avg_M, avg_A, admit_rate, s.get('miss_rate_avg')]
            row = [r if r is not None else 'NA' for r in row]
            print('| ' + ' | '.join(str(r) if not isinstance(r,float) else f'{r:.6g}' for r in row) + ' |')


DATASET: wiki2018
| Method | Cache | avg_M | avg_A | admit_rate | miss_rate_avg |
| --- | --- | --- | --- | --- | --- |
| baseline_lru | 16123 | NA | NA | NA | NA |
| baseline_lru | 20154 | NA | NA | NA | NA |
| baseline_lru | 40308 | NA | NA | NA | NA |
| baseline_lru | 60463 | NA | NA | NA | NA |
| baseline_lru | 80617 | NA | NA | NA | NA |
| baseline_lru | 100772 | NA | NA | NA | NA |
| baseline_lru-k | 16123 | NA | NA | NA | NA |
| baseline_lru-k | 20154 | NA | NA | NA | NA |
| baseline_lru-k | 40308 | NA | NA | NA | NA |
| baseline_lru-k | 60463 | NA | NA | NA | NA |
| baseline_lru-k | 80617 | NA | NA | NA | NA |
| baseline_lru-k | 100772 | NA | NA | NA | NA |
| ilnse_xu | 16123 | NA | NA | NA | NA |
| ilnse_xu | 20154 | NA | NA | NA | NA |
| ilnse_xu | 40308 | NA | NA | NA | NA |
| ilnse_xu | 60463 | NA | NA | NA | NA |
| ilnse_xu | 80617 | NA | NA | NA | NA |
| ilnse_xu | 100772 | NA | NA | NA | NA |
| gdbt_A2 | 16123 | NA | NA | NA | NA |
| gdbt_A2 | 20154 | NA | NA | NA | NA |

## (4) Drift-to-Budget Linkage (Spearman correlations)


In [9]:
for ds_name, ds_dir in DATASETS.items():
    records = load_latest_summaries(ds_dir)
    caches = get_caches(records)
    print('DATASET:', ds_name)
    header = ['Method','Cache','rho(drift,budget)','rho(drift,admit)']
    print('| ' + ' | '.join(header) + ' |')
    print('| ' + ' | '.join(['---']*len(header)) + ' |')
    for m in METHODS_MAIN:
        for c in caches:
            s = records.get((m,c), {})
            slot_path = s.get('slot_log_path')
            if not slot_path or not Path(slot_path).exists():
                continue
            drift=[]; budget=[]; admit=[]
            for row in iter_jsonl(Path(slot_path)):
                if row.get('phase')!='cache':
                    continue
                drift.append(row.get('drift_norm',0.0))
                budget.append(row.get('admit_budget',0))
                admit.append(row.get('admit_selected',0))
            rb = spearman(drift,budget)
            ra = spearman(drift,admit)
            print('| ' + ' | '.join([m,str(c), f'{rb:.4f}' if rb is not None else 'NA', f'{ra:.4f}' if ra is not None else 'NA']) + ' |')


DATASET: wiki2018
| Method | Cache | rho(drift,budget) | rho(drift,admit) |
| --- | --- | --- | --- |
| ilnse_A2_fixed | 16123 | NA | NA |
| ilnse_A2_fixed | 20154 | NA | NA |
| ilnse_A2_fixed | 40308 | NA | NA |
| ilnse_A2_fixed | 60463 | NA | NA |
| ilnse_A2_fixed | 80617 | NA | NA |
| ilnse_A2_fixed | 100772 | NA | NA |
| ilnse_A2_drift_only | 16123 | 0.9668 | 0.9668 |
| ilnse_A2_drift_only | 20154 | 0.9657 | 0.9657 |
| ilnse_A2_drift_only | 40308 | 0.9572 | 0.9572 |
| ilnse_A2_drift_only | 60463 | 0.9745 | 0.9745 |
| ilnse_A2_drift_only | 80617 | 0.9685 | 0.9685 |
| ilnse_A2_drift_only | 100772 | 0.9685 | 0.9685 |
| ilnse_A2_guard_only | 16123 | 0.0871 | 0.0871 |
| ilnse_A2_guard_only | 20154 | 0.0777 | 0.0777 |
| ilnse_A2_guard_only | 40308 | -0.0669 | -0.0669 |
| ilnse_A2_guard_only | 60463 | 0.2842 | 0.2842 |
| ilnse_A2_guard_only | 80617 | 0.2961 | 0.2961 |
| ilnse_A2_guard_only | 100772 | 0.3108 | 0.3108 |
| ilnse_A2_full | 16123 | 0.4755 | 0.4755 |
| ilnse_A2_full | 20154 | 0

## (5) Miss-pressure Effect (rho(miss_rate, pressure))


In [10]:
for ds_name, ds_dir in DATASETS.items():
    records = load_latest_summaries(ds_dir)
    caches = get_caches(records)
    print('DATASET:', ds_name)
    header = ['Method','Cache','rho(miss,pressure)']
    print('| ' + ' | '.join(header) + ' |')
    print('| ' + ' | '.join(['---']*len(header)) + ' |')
    for m in METHODS_MAIN:
        for c in caches:
            s = records.get((m,c), {})
            slot_path = s.get('slot_log_path')
            if not slot_path or not Path(slot_path).exists():
                continue
            miss=[]; pressure=[]
            for row in iter_jsonl(Path(slot_path)):
                if row.get('phase')!='cache':
                    continue
                miss.append(row.get('miss_rate',0.0))
                pressure.append(row.get('pressure_mult',0.0))
            rho = spearman(miss,pressure)
            print('| ' + ' | '.join([m,str(c), f'{rho:.4f}' if rho is not None else 'NA']) + ' |')


DATASET: wiki2018
| Method | Cache | rho(miss,pressure) |
| --- | --- | --- |
| ilnse_A2_fixed | 16123 | NA |
| ilnse_A2_fixed | 20154 | NA |
| ilnse_A2_fixed | 40308 | NA |
| ilnse_A2_fixed | 60463 | NA |
| ilnse_A2_fixed | 80617 | NA |
| ilnse_A2_fixed | 100772 | NA |
| ilnse_A2_drift_only | 16123 | 1.0000 |
| ilnse_A2_drift_only | 20154 | 1.0000 |
| ilnse_A2_drift_only | 40308 | 0.9913 |
| ilnse_A2_drift_only | 60463 | 0.9990 |
| ilnse_A2_drift_only | 80617 | 1.0000 |
| ilnse_A2_drift_only | 100772 | 1.0000 |
| ilnse_A2_guard_only | 16123 | 0.0224 |
| ilnse_A2_guard_only | 20154 | 0.0773 |
| ilnse_A2_guard_only | 40308 | 0.0868 |
| ilnse_A2_guard_only | 60463 | 0.0022 |
| ilnse_A2_guard_only | 80617 | -0.0330 |
| ilnse_A2_guard_only | 100772 | 0.0196 |
| ilnse_A2_full | 16123 | 0.0559 |
| ilnse_A2_full | 20154 | 0.0892 |
| ilnse_A2_full | 40308 | 0.0734 |
| ilnse_A2_full | 60463 | 0.0097 |
| ilnse_A2_full | 80617 | -0.0324 |
| ilnse_A2_full | 100772 | 0.0145 |
| ilnse_A0_full | 1612

## (6) Churn / Write Intensity (avg inserts/evictions)


In [11]:
for ds_name, ds_dir in DATASETS.items():
    records = load_latest_summaries(ds_dir)
    caches = get_caches(records)
    print('DATASET:', ds_name)
    header = ['Method','Cache','avg_inserts','avg_evictions']
    print('| ' + ' | '.join(header) + ' |')
    print('| ' + ' | '.join(['---']*len(header)) + ' |')
    for m in METHODS_MAIN:
        for c in caches:
            s = records.get((m,c), {})
            avg_ins = s.get('cache_inserts_avg')
            avg_evict = s.get('cache_evictions_avg')
            row = [m, c, avg_ins, avg_evict]
            row = [r if r is not None else 'NA' for r in row]
            print('| ' + ' | '.join(str(r) if not isinstance(r,float) else f'{r:.6g}' for r in row) + ' |')


DATASET: wiki2018
| Method | Cache | avg_inserts | avg_evictions |
| --- | --- | --- | --- |
| baseline_lru | 16123 | NA | NA |
| baseline_lru | 20154 | NA | NA |
| baseline_lru | 40308 | NA | NA |
| baseline_lru | 60463 | NA | NA |
| baseline_lru | 80617 | NA | NA |
| baseline_lru | 100772 | NA | NA |
| baseline_lru-k | 16123 | NA | NA |
| baseline_lru-k | 20154 | NA | NA |
| baseline_lru-k | 40308 | NA | NA |
| baseline_lru-k | 60463 | NA | NA |
| baseline_lru-k | 80617 | NA | NA |
| baseline_lru-k | 100772 | NA | NA |
| ilnse_xu | 16123 | NA | NA |
| ilnse_xu | 20154 | NA | NA |
| ilnse_xu | 40308 | NA | NA |
| ilnse_xu | 60463 | NA | NA |
| ilnse_xu | 80617 | NA | NA |
| ilnse_xu | 100772 | NA | NA |
| gdbt_A2 | 16123 | NA | NA |
| gdbt_A2 | 20154 | NA | NA |
| gdbt_A2 | 40308 | NA | NA |
| gdbt_A2 | 60463 | NA | NA |
| gdbt_A2 | 80617 | NA | NA |
| gdbt_A2 | 100772 | NA | NA |
| ilnse_A2_fixed | 16123 | 1275.67 | 1096.52 |
| ilnse_A2_fixed | 20154 | 1595.08 | 1371.14 |
| ilnse_A2_

## (7) Pollution Proxy (summary)


In [12]:
for ds_name, ds_dir in DATASETS.items():
    records = load_latest_summaries(ds_dir)
    caches = get_caches(records)
    print('DATASET:', ds_name)
    header = ['Method','Cache','pollution_rate','hit_yield','admission_precision_total']
    print('| ' + ' | '.join(header) + ' |')
    print('| ' + ' | '.join(['---']*len(header)) + ' |')
    for m in METHODS_MAIN:
        for c in caches:
            s = records.get((m,c), {})
            row = [m, c, s.get('pollution_rate_total'), s.get('hit_yield_total'), s.get('admission_precision_total')]
            row = [r if r is not None else 'NA' for r in row]
            print('| ' + ' | '.join(str(r) if not isinstance(r,float) else f'{r:.6g}' for r in row) + ' |')


DATASET: wiki2018
| Method | Cache | pollution_rate | hit_yield | admission_precision_total |
| --- | --- | --- | --- | --- |
| baseline_lru | 16123 | NA | NA | NA |
| baseline_lru | 20154 | NA | NA | NA |
| baseline_lru | 40308 | NA | NA | NA |
| baseline_lru | 60463 | NA | NA | NA |
| baseline_lru | 80617 | NA | NA | NA |
| baseline_lru | 100772 | NA | NA | NA |
| baseline_lru-k | 16123 | NA | NA | NA |
| baseline_lru-k | 20154 | NA | NA | NA |
| baseline_lru-k | 40308 | NA | NA | NA |
| baseline_lru-k | 60463 | NA | NA | NA |
| baseline_lru-k | 80617 | NA | NA | NA |
| baseline_lru-k | 100772 | NA | NA | NA |
| ilnse_xu | 16123 | NA | NA | NA |
| ilnse_xu | 20154 | NA | NA | NA |
| ilnse_xu | 40308 | NA | NA | NA |
| ilnse_xu | 60463 | NA | NA | NA |
| ilnse_xu | 80617 | NA | NA | NA |
| ilnse_xu | 100772 | NA | NA | NA |
| gdbt_A2 | 16123 | NA | NA | NA |
| gdbt_A2 | 20154 | NA | NA | NA |
| gdbt_A2 | 40308 | NA | NA | NA |
| gdbt_A2 | 60463 | NA | NA | NA |
| gdbt_A2 | 80617 | NA 

## (8) Eviction Harm Proxy (NA unless logged)
This matrix is not available in current logs.


## (9) Lag‑Aware Precision Summary


In [13]:
TARGET_DEFAULT = 0.12
for ds_name, ds_dir in DATASETS.items():
    records = load_latest_summaries(ds_dir)
    caches = get_caches(records)
    print('DATASET:', ds_name)
    header = ['Method','Cache','prec_eff_mean','prec_eff_median','frac_below_target']
    print('| ' + ' | '.join(header) + ' |')
    print('| ' + ' | '.join(['---']*len(header)) + ' |')
    for m in METHODS_MAIN:
        for c in caches:
            s = records.get((m,c), {})
            slot_path = s.get('slot_log_path')
            if not slot_path or not Path(slot_path).exists():
                continue
            vals=[]
            for row in iter_jsonl(Path(slot_path)):
                if row.get('phase')!='cache':
                    continue
                v = row.get('admission_precision_eff')
                if v is not None:
                    vals.append(v)
            if not vals:
                continue
            target = s.get('admission_precision_target') or TARGET_DEFAULT
            mean = statistics.mean(vals)
            median = statistics.median(vals)
            frac = sum(1 for v in vals if v < target)/len(vals)
            print('| ' + ' | '.join([m,str(c), f'{mean:.6g}', f'{median:.6g}', f'{frac:.3f}']) + ' |')


DATASET: wiki2018
| Method | Cache | prec_eff_mean | prec_eff_median | frac_below_target |
| --- | --- | --- | --- | --- |
| ilnse_A2_fixed | 16123 | 0.200688 | 0.146512 | 0.090 |
| ilnse_A2_fixed | 20154 | 0.159727 | 0.108493 | 0.697 |
| ilnse_A2_fixed | 40308 | 0.0873443 | 0.0493023 | 0.888 |
| ilnse_A2_fixed | 60463 | 0.0639762 | 0.036172 | 0.944 |
| ilnse_A2_fixed | 80617 | 0.0514171 | 0.0288372 | 0.955 |
| ilnse_A2_fixed | 100772 | 0.0431028 | 0.0231952 | 0.955 |
| ilnse_A2_drift_only | 16123 | 0.200879 | 0.145031 | 0.079 |
| ilnse_A2_drift_only | 20154 | 0.165022 | 0.111898 | 0.607 |
| ilnse_A2_drift_only | 40308 | 0.0779764 | 0.0529833 | 0.910 |
| ilnse_A2_drift_only | 60463 | 0.0536025 | 0.0401163 | 0.966 |
| ilnse_A2_drift_only | 80617 | 0.0415058 | 0.0303474 | 0.966 |
| ilnse_A2_drift_only | 100772 | 0.0334521 | 0.0239145 | 0.966 |
| ilnse_A2_guard_only | 16123 | 0.189885 | 0.146127 | 0.135 |
| ilnse_A2_guard_only | 20154 | 0.155721 | 0.112445 | 0.685 |
| ilnse_A2_guard_only 

## (10) Precision‑to‑Throttle Check (using score_gate_applied)


In [14]:
for ds_name, ds_dir in DATASETS.items():
    records = load_latest_summaries(ds_dir)
    caches = get_caches(records)
    print('DATASET:', ds_name)
    header = ['Method','Cache','avg_M_lowPrec','avg_M_highPrec','gate_rate']
    print('| ' + ' | '.join(header) + ' |')
    print('| ' + ' | '.join(['---']*len(header)) + ' |')
    for m in METHODS_MAIN:
        for c in caches:
            s = records.get((m,c), {})
            slot_path = s.get('slot_log_path')
            if not slot_path or not Path(slot_path).exists():
                continue
            low=[]; high=[]; gates=0; n=0
            target = s.get('admission_precision_target') or 0.12
            for row in iter_jsonl(Path(slot_path)):
                if row.get('phase')!='cache':
                    continue
                prec = row.get('admission_precision_eff')
                budget = row.get('admit_budget')
                if prec is None or budget is None:
                    continue
                if prec < target:
                    low.append(budget)
                else:
                    high.append(budget)
                if row.get('score_gate_applied'):
                    gates += 1
                n += 1
            if n == 0:
                continue
            avg_low = statistics.mean(low) if low else None
            avg_high = statistics.mean(high) if high else None
            gate_rate = gates/n if n else None
            row = [m, c, avg_low, avg_high, gate_rate]
            row = [r if r is not None else 'NA' for r in row]
            print('| ' + ' | '.join(str(r) if not isinstance(r,float) else f'{r:.6g}' for r in row) + ' |')


DATASET: wiki2018
| Method | Cache | avg_M_lowPrec | avg_M_highPrec | gate_rate |
| --- | --- | --- | --- | --- |
| ilnse_A2_fixed | 16123 | 1290 | 1290 | 0 |
| ilnse_A2_fixed | 20154 | 1613 | 1613 | 0 |
| ilnse_A2_fixed | 40308 | 3225 | 3225 | 0 |
| ilnse_A2_fixed | 60463 | 4838 | 4838 | 0 |
| ilnse_A2_fixed | 80617 | 6450 | 6450 | 0 |
| ilnse_A2_fixed | 100772 | 8062 | 8062 | 0 |
| ilnse_A2_drift_only | 16123 | 1151.43 | 1131.1 | 0 |
| ilnse_A2_drift_only | 20154 | 1357.59 | 1479.46 | 0 |
| ilnse_A2_drift_only | 40308 | 4423.75 | 4812.5 | 0 |
| ilnse_A2_drift_only | 60463 | 8965.71 | 11742.3 | 0 |
| ilnse_A2_drift_only | 80617 | 11939.8 | 15775.7 | 0 |
| ilnse_A2_drift_only | 100772 | 14848.8 | 19651 | 0 |
| ilnse_A2_guard_only | 16123 | 623.417 | 1081.74 | 0.0337079 |
| ilnse_A2_guard_only | 20154 | 816.721 | 1663.29 | 0.0337079 |
| ilnse_A2_guard_only | 40308 | 2163.07 | 5091.38 | 0.640449 |
| ilnse_A2_guard_only | 60463 | 2190.33 | 6534.33 | 0.921348 |
| ilnse_A2_guard_only | 8061

## (11) Score‑Quality Summary (spread & quality_mult)


In [15]:
def quantiles(vals, qs):
    if not vals:
        return [None]*len(qs)
    vals = sorted(vals)
    out=[]
    for q in qs:
        idx = int(round(q*(len(vals)-1)))
        out.append(vals[idx])
    return out

for ds_name, ds_dir in DATASETS.items():
    records = load_latest_summaries(ds_dir)
    caches = get_caches(records)
    print('DATASET:', ds_name)
    header = ['Method','Cache','spread_mean','q10','q90','quality_mean','q10','q90']
    print('| ' + ' | '.join(header) + ' |')
    print('| ' + ' | '.join(['---']*len(header)) + ' |')
    for m in METHODS_MAIN:
        for c in caches:
            s = records.get((m,c), {})
            slot_path = s.get('slot_log_path')
            if not slot_path or not Path(slot_path).exists():
                continue
            spreads=[]; quals=[]
            for row in iter_jsonl(Path(slot_path)):
                if row.get('phase')!='cache':
                    continue
                if row.get('score_spread') is not None:
                    spreads.append(row.get('score_spread'))
                if row.get('quality_mult') is not None:
                    quals.append(row.get('quality_mult'))
            if not spreads and not quals:
                continue
            sp_mean = statistics.mean(spreads) if spreads else None
            sp_q10, sp_q90 = quantiles(spreads, [0.1,0.9]) if spreads else (None,None)
            q_mean = statistics.mean(quals) if quals else None
            q_q10, q_q90 = quantiles(quals, [0.1,0.9]) if quals else (None,None)
            row = [m, c, sp_mean, sp_q10, sp_q90, q_mean, q_q10, q_q90]
            row = [r if r is not None else 'NA' for r in row]
            print('| ' + ' | '.join(str(r) if not isinstance(r,float) else f'{r:.6g}' for r in row) + ' |')


DATASET: wiki2018
| Method | Cache | spread_mean | q10 | q90 | quality_mean | q10 | q90 |
| --- | --- | --- | --- | --- | --- | --- | --- |
| ilnse_A2_fixed | 16123 | 0.0722463 | 0.000715286 | 0.342411 | 0.911764 | 0.76 | 1.17679 |
| ilnse_A2_fixed | 20154 | 0.0696107 | 0.000707111 | 0.330574 | 0.910081 | 0.76 | 1.1767 |
| ilnse_A2_fixed | 40308 | 0.0308165 | 0.000559535 | 0.112787 | 0.891955 | 0.729248 | 1.06993 |
| ilnse_A2_fixed | 60463 | 0.0247578 | 0.000125613 | 0.111489 | 0.900286 | 0.7 | 1.20153 |
| ilnse_A2_fixed | 80617 | 0.02259 | 8.96e-05 | 0.108638 | 0.897667 | 0.7 | 1.1869 |
| ilnse_A2_fixed | 100772 | 0.0223448 | 4.15392e-05 | 0.108634 | 0.903883 | 0.7 | 1.3 |
| ilnse_A2_drift_only | 16123 | 0.0718968 | 0.000714848 | 0.342402 | 0.911633 | 0.76 | 1.17604 |
| ilnse_A2_drift_only | 20154 | 0.0677492 | 0.000706241 | 0.300688 | 0.911313 | 0.76 | 1.1942 |
| ilnse_A2_drift_only | 40308 | 0.0273084 | 0.000576398 | 0.111502 | 0.887368 | 0.729248 | 1.07018 |
| ilnse_A2_drift_only |

## (12) Score‑Quality vs Admission (Spearman)


In [16]:
for ds_name, ds_dir in DATASETS.items():
    records = load_latest_summaries(ds_dir)
    caches = get_caches(records)
    print('DATASET:', ds_name)
    header = ['Method','Cache','rho(quality,admit)']
    print('| ' + ' | '.join(header) + ' |')
    print('| ' + ' | '.join(['---']*len(header)) + ' |')
    for m in METHODS_MAIN:
        for c in caches:
            s = records.get((m,c), {})
            slot_path = s.get('slot_log_path')
            if not slot_path or not Path(slot_path).exists():
                continue
            q=[]; a=[]
            for row in iter_jsonl(Path(slot_path)):
                if row.get('phase')!='cache':
                    continue
                if row.get('quality_mult') is None:
                    continue
                q.append(row.get('quality_mult'))
                a.append(row.get('admit_selected',0))
            rho = spearman(q,a)
            print('| ' + ' | '.join([m,str(c), f'{rho:.4f}' if rho is not None else 'NA']) + ' |')


DATASET: wiki2018
| Method | Cache | rho(quality,admit) |
| --- | --- | --- |
| ilnse_A2_fixed | 16123 | NA |
| ilnse_A2_fixed | 20154 | NA |
| ilnse_A2_fixed | 40308 | NA |
| ilnse_A2_fixed | 60463 | NA |
| ilnse_A2_fixed | 80617 | NA |
| ilnse_A2_fixed | 100772 | NA |
| ilnse_A2_drift_only | 16123 | -0.1162 |
| ilnse_A2_drift_only | 20154 | -0.1362 |
| ilnse_A2_drift_only | 40308 | -0.4320 |
| ilnse_A2_drift_only | 60463 | -0.4215 |
| ilnse_A2_drift_only | 80617 | -0.4545 |
| ilnse_A2_drift_only | 100772 | -0.4849 |
| ilnse_A2_guard_only | 16123 | 0.6947 |
| ilnse_A2_guard_only | 20154 | 0.7955 |
| ilnse_A2_guard_only | 40308 | 0.3060 |
| ilnse_A2_guard_only | 60463 | -0.1818 |
| ilnse_A2_guard_only | 80617 | -0.3167 |
| ilnse_A2_guard_only | 100772 | -0.3200 |
| ilnse_A2_full | 16123 | 0.4715 |
| ilnse_A2_full | 20154 | 0.5328 |
| ilnse_A2_full | 40308 | -0.0779 |
| ilnse_A2_full | 60463 | -0.1436 |
| ilnse_A2_full | 80617 | -0.3118 |
| ilnse_A2_full | 100772 | -0.3284 |
| ilnse_A0_

## (13) Contribution Ablation Table


In [17]:
for ds_name, ds_dir in DATASETS.items():
    records = load_latest_summaries(ds_dir)
    caches = get_caches(records)
    print('DATASET:', ds_name)
    header = ['Variant','avg_HR','avg_pollution','avg_admit_rate','prec_eff_mean','gate_rate']
    print('| ' + ' | '.join(header) + ' |')
    print('| ' + ' | '.join(['---']*len(header)) + ' |')
    for m in ABLAT:
        hrs=[]; poll=[]; rates=[]; precs=[]; gates=[]; total_gates=0; total_slots=0
        for c in caches:
            s = records.get((m,c), {})
            if not s: continue
            hrs.append(s.get('hit_ratio'))
            if s.get('pollution_rate_total') is not None: poll.append(s.get('pollution_rate_total'))
            mc = s.get('miss_candidates_total') or 0
            if mc > 0: rates.append(s.get('admit_selected_total',0)/mc)
            if s.get('admission_precision_eff_avg') is not None: precs.append(s.get('admission_precision_eff_avg'))
            slot_path = s.get('slot_log_path')
            if slot_path and Path(slot_path).exists():
                for row in iter_jsonl(Path(slot_path)):
                    if row.get('phase')!='cache':
                        continue
                    total_slots += 1
                    if row.get('score_gate_applied'): total_gates += 1
        avg_hr = statistics.mean([h for h in hrs if h is not None]) if hrs else None
        avg_poll = statistics.mean(poll) if poll else None
        avg_rate = statistics.mean(rates) if rates else None
        avg_prec = statistics.mean(precs) if precs else None
        gate_rate = (total_gates/total_slots) if total_slots else None
        row = [m, avg_hr, avg_poll, avg_rate, avg_prec, gate_rate]
        row = [r if r is not None else 'NA' for r in row]
        print('| ' + ' | '.join(str(r) if not isinstance(r,float) else f'{r:.6g}' for r in row) + ' |')


DATASET: wiki2018
| Variant | avg_HR | avg_pollution | avg_admit_rate | prec_eff_mean | gate_rate |
| --- | --- | --- | --- | --- | --- |
| ilnse_A2_fixed | 0.528405 | 0.633145 | 0.1004 | 0.101043 | 0 |
| ilnse_A2_drift_only | 0.523935 | 0.637996 | 0.169496 | 0.0954064 | 0 |
| ilnse_A2_guard_only | 0.537542 | 0.616062 | 0.0489431 | 0.0943875 | 0.572222 |
| ilnse_A2_full | 0.537407 | 0.620701 | 0.0533187 | 0.0923667 | 0.637037 |
DATASET: wikipedia_september_2007
| Variant | avg_HR | avg_pollution | avg_admit_rate | prec_eff_mean | gate_rate |
| --- | --- | --- | --- | --- | --- |
| ilnse_A2_fixed | 0.67078 | 0.687472 | 0.10821 | 0.0787049 | 0 |
| ilnse_A2_drift_only | 0.667754 | 0.695621 | 0.177727 | 0.0695772 | 0 |
| ilnse_A2_guard_only | 0.676564 | 0.658424 | 0.0494376 | 0.0711377 | 0.591667 |
| ilnse_A2_full | 0.676743 | 0.667946 | 0.0525977 | 0.0669691 | 0.614583 |


## (14) Feature Ablation (A0 vs A2)


In [18]:
for ds_name, ds_dir in DATASETS.items():
    records = load_latest_summaries(ds_dir)
    caches = get_caches(records)
    print('DATASET:', ds_name)
    header = ['Cache','HR(A0)','HR(A2)','ΔHR(A0-A2)']
    print('| ' + ' | '.join(header) + ' |')
    print('| ' + ' | '.join(['---']*len(header)) + ' |')
    for c in caches:
        a0 = records.get(('ilnse_A0_full',c), {}).get('hit_ratio')
        a2 = records.get(('ilnse_A2_full',c), {}).get('hit_ratio')
        d = (a0 - a2) if (a0 is not None and a2 is not None) else None
        print('| ' + ' | '.join([str(c), f'{a0:.6f}' if a0 is not None else 'NA', f'{a2:.6f}' if a2 is not None else 'NA', f'{d:.6f}' if d is not None else 'NA']) + ' |')


DATASET: wiki2018
| Cache | HR(A0) | HR(A2) | ΔHR(A0-A2) |
| --- | --- | --- | --- |
| 16123 | 0.434173 | 0.466867 | -0.032694 |
| 20154 | 0.467405 | 0.480824 | -0.013419 |
| 40308 | 0.525184 | 0.526211 | -0.001027 |
| 60463 | 0.562923 | 0.559610 | 0.003313 |
| 80617 | 0.587955 | 0.585296 | 0.002659 |
| 100772 | 0.606730 | 0.605636 | 0.001095 |
DATASET: wikipedia_september_2007
| Cache | HR(A0) | HR(A2) | ΔHR(A0-A2) |
| --- | --- | --- | --- |
| 12221 | 0.621851 | 0.636962 | -0.015112 |
| 15277 | 0.629776 | 0.644132 | -0.014356 |
| 30554 | 0.659248 | 0.670513 | -0.011265 |
| 45831 | 0.683880 | 0.689720 | -0.005840 |
| 61109 | 0.700599 | 0.704212 | -0.003613 |
| 76386 | 0.711882 | 0.714920 | -0.003038 |


## (15) Overhead (avg update/score time)


In [19]:
for ds_name, ds_dir in DATASETS.items():
    records = load_latest_summaries(ds_dir)
    caches = get_caches(records)
    print('DATASET:', ds_name)
    header = ['Method','Cache','avg_update_time_s','avg_score_time_s']
    print('| ' + ' | '.join(header) + ' |')
    print('| ' + ' | '.join(['---']*len(header)) + ' |')
    for m in METHODS_MAIN:
        for c in caches:
            s = records.get((m,c), {})
            row = [m, c, s.get('avg_update_time_s'), s.get('avg_score_time_s')]
            row = [r if r is not None else 'NA' for r in row]
            print('| ' + ' | '.join(str(r) if not isinstance(r,float) else f'{r:.6g}' for r in row) + ' |')


DATASET: wiki2018
| Method | Cache | avg_update_time_s | avg_score_time_s |
| --- | --- | --- | --- |
| baseline_lru | 16123 | NA | NA |
| baseline_lru | 20154 | NA | NA |
| baseline_lru | 40308 | NA | NA |
| baseline_lru | 60463 | NA | NA |
| baseline_lru | 80617 | NA | NA |
| baseline_lru | 100772 | NA | NA |
| baseline_lru-k | 16123 | NA | NA |
| baseline_lru-k | 20154 | NA | NA |
| baseline_lru-k | 40308 | NA | NA |
| baseline_lru-k | 60463 | NA | NA |
| baseline_lru-k | 80617 | NA | NA |
| baseline_lru-k | 100772 | NA | NA |
| ilnse_xu | 16123 | NA | NA |
| ilnse_xu | 20154 | NA | NA |
| ilnse_xu | 40308 | NA | NA |
| ilnse_xu | 60463 | NA | NA |
| ilnse_xu | 80617 | NA | NA |
| ilnse_xu | 100772 | NA | NA |
| gdbt_A2 | 16123 | NA | NA |
| gdbt_A2 | 20154 | NA | NA |
| gdbt_A2 | 40308 | NA | NA |
| gdbt_A2 | 60463 | NA | NA |
| gdbt_A2 | 80617 | NA | NA |
| gdbt_A2 | 100772 | NA | NA |
| ilnse_A2_fixed | 16123 | 0.619728 | 0.286461 |
| ilnse_A2_fixed | 20154 | 0.626856 | 0.283446 

## (16) Model Size / Learner Count (NA unless logged)
This matrix is not available in current logs.
